
# QML-SleepNet — Promoted Final 90.8627% Stage-09 + Stage-10 Refresh v1

## Purpose

This notebook performs the **minimum scientifically required refresh** after promoting the already-frozen fixed fusion:

\[
\textbf{25% Stage15A physiology + 75% guide-primary QML}
\]

in **logit space**, with fixed threshold 0.5.

The promoted frozen official-x artifact must have SHA-256:

`063a017e61188393bcdcdacb72958ffa3e7e0efa9432d33aa0845983462dfa1f`

and must reproduce official-x accuracy:

\[
\boxed{90.86270871985158\%}
\]

## Non-negotiable boundaries

- **ZERO training**
- **ZERO fine-tuning**
- **ZERO model selection**
- **ZERO threshold search**
- **ZERO fusion-weight search**
- **ZERO HMM refit**
- **ZERO calibration refit**
- **ZERO attempt to improve the headline metric**
- Existing Stage-09 QML-component XAI is reused only because the QML parent is byte-for-byte unchanged.
- Existing Stage-10 perturbation caches are reused only for the exact previously evaluated Stage06 temporal-channel perturbation.
- The classical physiology parent is held clean during those inherited ECG perturbations; therefore this notebook **does not claim full dual-parent end-to-end raw-ECG perturbation testing**.
- Sleep-stage robustness remains **not evaluable** because the frozen artifact contract contains no sleep-stage target.
- No quantum-advantage claim.

## Output folders

- `outputs/GUIDE_EXACT_METRICMAX/XAI_CAUSAL_INTERPRETABILITY_90P8627_PROMOTED_FINAL_v1`
- `outputs/GUIDE_EXACT_METRICMAX/STAGE10_MODEL_ROBUSTNESS_90P8627_PROMOTED_FINAL_v1`

If every fail-closed check passes, the next roadmap stage is:

**Stage 11 — Final Evaluation Protocol / Statistical Comparison**


In [ ]:

# Cell 1 — Drive, imports, immutable paths

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, uuid, warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    confusion_matrix, brier_score_loss, log_loss
)

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path("/content/drive/MyDrive/QML_SleepNet")
GX = ROOT / "outputs/GUIDE_EXACT_METRICMAX"

# Promoted final fusion.
FUSION_ROOT = GX / "FINAL_METRIC_CHAMPION_FIXED_FUSION_AUDIT_v1"
FUSED_X = FUSION_ROOT / "FINAL_FIXED_FUSION_X_PREDICTIONS_FROZEN_BEFORE_SCORING.npz"
FUSION_FREEZE = FUSION_ROOT / "FINAL_FIXED_FUSION_PRE_SCORE_FREEZE_MANIFEST.json"
FUSION_REPORT = FUSION_ROOT / "FINAL_FIXED_FUSION_POSTHOC_X_METRICS.json"
FUSION_DECISION = FUSION_ROOT / "FINAL_FIXED_FUSION_DEVELOPMENT_DECISION.json"
FUSION_BOOTSTRAP = FUSION_ROOT / "FIXED_FUSION_STRICT_GROUP_BOOTSTRAP.csv"

# Frozen parent systems.
PHYS_ROOT = GX / "DEV35_STAGE15A_PHYSIO_BOOSTING_CHALLENGER_v1_2_COVERAGESAFE"
PHYS_X = PHYS_ROOT / "DEV35_STAGE15A_PHYSIO_BOOSTING_X_PREDICTIONS_FROZEN_BEFORE_SCORING.npz"
PHYS_FREEZE = PHYS_ROOT / "DEV35_STAGE15A_PHYSIO_BOOSTING_PRE_SCORE_FREEZE_MANIFEST.json"
PHYS_MODELS = PHYS_ROOT / "final_models"

QML_ROOT = GX / "FINAL_FIXED_EQUAL_LOGIT_ENSEMBLE_v1"
QML_X = QML_ROOT / "FINAL_FIXED_EQUAL_LOGIT_X_PREDICTIONS_FROZEN_BEFORE_SCORING.npz"
QML_X_METRICS = QML_ROOT / "FINAL_FIXED_EQUAL_LOGIT_POSTHOC_X_METRICS.json"
QML_X_PRED_CSV = QML_ROOT / "FINAL_FIXED_EQUAL_LOGIT_POSTHOC_X_PREDICTIONS.csv"

# Existing completed Stage-09 / Stage-10 evidence for the unchanged QML parent.
OLD_XAI = GX / "XAI_CAUSAL_INTERPRETABILITY_90P1148_FINAL_v1_2"
OLD_XAI_MANIFEST = OLD_XAI / "XAI_FINAL_MANIFEST.json"

OLD_S10 = GX / "STAGE10_MODEL_ROBUSTNESS_FINAL_v1"
OLD_S10_MANIFEST = OLD_S10 / "STAGE10_FINAL_MANIFEST.json"
OLD_S10_CACHE = OLD_S10 / "cache"
OLD_S10_QUALITY = OLD_S10 / "STAGE10_ECG_QUALITY_PROXY.csv"
OLD_S10_ARTIFACT = OLD_S10 / "STAGE10_ARTIFACT_SEGMENT_REPORT.json"
OLD_S10_SENS = OLD_S10 / "STAGE10_INDIVIDUAL_FEATURE_ZEROOUT_SENSITIVITY.csv"
OLD_S10_UNSUPPORTED = OLD_S10 / "STAGE10_UNSUPPORTED_GUIDE_ITEMS.json"
OLD_S10_PERT = OLD_S10 / "STAGE10_PERTURBATION_ROBUSTNESS.csv"

# Task-C metadata for the same x01-x35 records.
TASKC_ROOT = GX / "STAGE06_TASKC_FINAL_ABC_FROM_FROZEN_TASKA_v1"
TASKC_RECORDS = TASKC_ROOT / "TASKC_ABC_PER_RECORD_FINAL_RESULTS.csv"

# Stage15A feature schema for classical-parent model explanation.
A15_ROOT = (
    ROOT / "outputs/guide_pipeline/15A_feature_bank_v1_4_3"
    / "d19e39dac5d0_7b0087f85784_154dd6572ddd"
)
A15_SCHEMA = A15_ROOT / "feature_schema.json"

# New promoted-final evidence folders.
XAI_OUT = GX / "XAI_CAUSAL_INTERPRETABILITY_90P8627_PROMOTED_FINAL_v1"
S10_OUT = GX / "STAGE10_MODEL_ROBUSTNESS_90P8627_PROMOTED_FINAL_v1"
XAI_OUT.mkdir(parents=True, exist_ok=True)
S10_OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_FUSED_SHA = "063a017e61188393bcdcdacb72958ffa3e7e0efa9432d33aa0845983462dfa1f"
EXPECTED_QML_SHA = "f121a79191be00a28f33e06e7dec20cc689268b10a988c52e21284c90d1e2eef"
EXPECTED_PHYS_SHA = "e548af5c1d8ad39f8ea192680ee7ae7af303b993e65f410ea2ee18f015e3e244"
EXPECTED_ACC = 0.9086270871985158
EXPECTED_W_PHYS = 0.25
EXPECTED_W_QML = 0.75
EXPECTED_THRESHOLD = 0.5
EPS = 1e-8

required = [
    FUSED_X, FUSION_FREEZE, FUSION_REPORT, FUSION_DECISION, FUSION_BOOTSTRAP,
    PHYS_X, PHYS_FREEZE, QML_X, QML_X_METRICS, QML_X_PRED_CSV,
    OLD_XAI_MANIFEST, OLD_S10_MANIFEST, OLD_S10_QUALITY, OLD_S10_ARTIFACT,
    OLD_S10_SENS, OLD_S10_UNSUPPORTED, OLD_S10_PERT, TASKC_RECORDS, A15_SCHEMA
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing required frozen artifact(s):\n" + "\n".join(missing))

print("Promoted-final refresh initialized.")
print("ZERO TRAINING / ZERO TUNING / ZERO MODEL SELECTION")
print("XAI output:", XAI_OUT)
print("Stage10 output:", S10_OUT)


In [ ]:

# Cell 2 — deterministic helpers

def sha256_file(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_name(path.name + f".tmp.{uuid.uuid4().hex}")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True, default=str))
    os.replace(tmp, path)

def atomic_csv(path, df):
    path = Path(path)
    tmp = path.with_name(path.name + f".tmp.{uuid.uuid4().hex}")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def logit(p):
    p = np.clip(np.asarray(p, np.float64), EPS, 1-EPS)
    return np.log(p) - np.log1p(-p)

def sigmoid(z):
    z = np.asarray(z, np.float64)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

def fixed_logit_fusion(p_phys, p_qml, w_phys=EXPECTED_W_PHYS):
    return sigmoid(float(w_phys)*logit(p_phys) + (1.0-float(w_phys))*logit(p_qml))

def equal_logit_mean_n(*scores):
    return sigmoid(np.mean(np.column_stack([logit(s) for s in scores]), axis=1))

def metrics(y, p, threshold=0.5):
    y = np.asarray(y, np.int8)
    p = np.asarray(p, np.float64)
    pred = (p >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {
        "n": int(len(y)),
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "sensitivity": float(recall_score(y, pred, zero_division=0)),
        "specificity": float(tn/max(tn+fp,1)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "mcc": float(matthews_corrcoef(y, pred)),
        "auroc": float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else np.nan,
        "auprc": float(average_precision_score(y, p)) if len(np.unique(y)) == 2 else np.nan,
        "brier": float(brier_score_loss(y, p)),
        "nll": float(log_loss(y, np.column_stack([1-p,p]), labels=[0,1])),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def parse_uid(uid):
    r, e = str(uid).rsplit(":", 1)
    return r, int(e)

print("Helpers ready.")


In [ ]:

# Cell 3 — FAIL-CLOSED promoted-final provenance and exact 90.8627% reproduction

freeze = json.loads(FUSION_FREEZE.read_text())
report = json.loads(FUSION_REPORT.read_text())
decision = json.loads(FUSION_DECISION.read_text())
phys_freeze = json.loads(PHYS_FREEZE.read_text())
old_xai_manifest = json.loads(OLD_XAI_MANIFEST.read_text())
old_s10_manifest = json.loads(OLD_S10_MANIFEST.read_text())

# Provenance invariants.
if sha256_file(FUSED_X) != EXPECTED_FUSED_SHA:
    raise RuntimeError("PROMOTED FINAL SHA MISMATCH")
if freeze.get("fused_prediction_sha256") != EXPECTED_FUSED_SHA:
    raise RuntimeError("Freeze manifest fused SHA mismatch")
if freeze.get("qml_component_x_sha256") != EXPECTED_QML_SHA:
    raise RuntimeError("Freeze manifest QML parent SHA mismatch")
if freeze.get("stage15a_component_x_sha256") != EXPECTED_PHYS_SHA:
    raise RuntimeError("Freeze manifest physiology parent SHA mismatch")
if sha256_file(QML_X) != EXPECTED_QML_SHA:
    raise RuntimeError("QML parent file SHA mismatch")
if sha256_file(PHYS_X) != EXPECTED_PHYS_SHA:
    raise RuntimeError("Physiology parent file SHA mismatch")
if freeze.get("official_x_labels_loaded") is not False:
    raise RuntimeError("Fusion selection provenance failure: x labels were loaded")
if freeze.get("training_performed") is not False:
    raise RuntimeError("Fusion audit unexpectedly trained a model")
if freeze.get("threshold_search") is not False:
    raise RuntimeError("Fusion audit unexpectedly searched threshold")
if freeze.get("calibration_refit") is not False or freeze.get("hmm_refit") is not False:
    raise RuntimeError("Fusion audit unexpectedly refit calibration/HMM")

win = freeze["winning_fusion"]
if win.get("candidate") != "FIXED_LOGIT_PHYS_25_QML_75":
    raise RuntimeError("Unexpected promoted fusion identity")
if abs(float(win["w_phys"]) - EXPECTED_W_PHYS) > 1e-15:
    raise RuntimeError("Physiology weight drift")
if abs(float(win["w_qml"]) - EXPECTED_W_QML) > 1e-15:
    raise RuntimeError("QML weight drift")
if decision.get("promote_fusion") is not True:
    raise RuntimeError("Development promotion decision is not TRUE")
if win.get("promotion_gate_pass") is not True:
    raise RuntimeError("Promoted fusion did not pass the declared development gate")

# Existing component evidence must be for the exact unchanged QML parent.
if old_xai_manifest.get("final_taska_prediction_sha256") != EXPECTED_QML_SHA:
    raise RuntimeError("Inherited Stage09 XAI is not bound to the QML parent SHA")
if old_s10_manifest.get("primary_frozen_prediction_sha256") != EXPECTED_QML_SHA:
    raise RuntimeError("Inherited Stage10 evidence is not bound to the QML parent SHA")

# Load promoted artifact.
fz = np.load(FUSED_X, allow_pickle=False)
required_keys = {
    "test_uids", "physiology_hmm_posterior", "qml_core3_hmm_logit_mean",
    "physiology_weight", "qml_weight", "fused_probability", "prediction",
    "hard_threshold"
}
if not required_keys.issubset(fz.files):
    raise RuntimeError(f"Promoted NPZ missing keys: {required_keys-set(fz.files)}")

UID = np.asarray(fz["test_uids"]).astype(str)
P_PHYS = np.asarray(fz["physiology_hmm_posterior"], np.float64)
P_QML = np.asarray(fz["qml_core3_hmm_logit_mean"], np.float64)
P_FINAL = np.asarray(fz["fused_probability"], np.float64)
PRED_FINAL = np.asarray(fz["prediction"], np.int8)
W_PHYS = float(np.asarray(fz["physiology_weight"]).item())
W_QML = float(np.asarray(fz["qml_weight"]).item())
THR = float(np.asarray(fz["hard_threshold"]).item())

if len(UID) != 17248:
    raise RuntimeError(f"Expected 17,248 promoted rows, got {len(UID)}")
if not (np.isfinite(P_PHYS).all() and np.isfinite(P_QML).all() and np.isfinite(P_FINAL).all()):
    raise RuntimeError("Non-finite promoted-parent/final probability")
if abs(W_PHYS-EXPECTED_W_PHYS)>1e-15 or abs(W_QML-EXPECTED_W_QML)>1e-15:
    raise RuntimeError("Stored fusion weight drift")
if abs(THR-EXPECTED_THRESHOLD)>1e-15:
    raise RuntimeError("Stored threshold drift")

RECON = fixed_logit_fusion(P_PHYS, P_QML, W_PHYS)
recon_max = float(np.max(np.abs(RECON-P_FINAL)))
if recon_max > 1e-12:
    raise RuntimeError(f"Exact fixed-fusion reconstruction failed: {recon_max}")
if not np.array_equal(PRED_FINAL, (P_FINAL >= THR).astype(np.int8)):
    raise RuntimeError("Promoted prediction does not match promoted score/threshold")

# Reuse the already-opened official-x y_true table from the canonical QML scorer.
pred_df = pd.read_csv(QML_X_PRED_CSV)
pred_df["uid"] = pred_df["uid"].astype(str)
if not np.array_equal(pred_df["uid"].to_numpy(), UID):
    raise RuntimeError("Official-x scoring UID order differs from promoted frozen artifact")
Y = np.asarray(pred_df["y_true"], np.int8)

M_FINAL = metrics(Y, P_FINAL, THR)
if abs(M_FINAL["accuracy"] - EXPECTED_ACC) > 1e-12:
    raise RuntimeError(f"90.8627% reproduction failed: {M_FINAL['accuracy']}")
if abs(M_FINAL["accuracy"] - float(report["fusion"]["accuracy"])) > 1e-12:
    raise RuntimeError("Saved fusion report disagrees with reproduced result")

REC = np.asarray([parse_uid(u)[0] for u in UID])
EP = np.asarray([parse_uid(u)[1] for u in UID], np.int64)

print("="*110)
print("PROMOTED FINAL PROVENANCE: PASS")
print("="*110)
print("SHA:", EXPECTED_FUSED_SHA)
print("Weights: physiology =", W_PHYS, "QML =", W_QML)
print("Exact reconstruction max |Δ|:", recon_max)
print(json.dumps(M_FINAL, indent=2))
print("\nLOCKED PROMOTED OFFICIAL-X ACCURACY:", 100*M_FINAL["accuracy"])
print("Training/tuning/model selection in this refresh: NO")


In [ ]:

# Cell 4 — Stage 09 refresh: exact top-level parent contribution + deterministic representative cases

L_PHYS = W_PHYS * logit(P_PHYS)
L_QML = W_QML * logit(P_QML)
L_SUM = L_PHYS + L_QML

if np.max(np.abs(sigmoid(L_SUM) - P_FINAL)) > 1e-12:
    raise RuntimeError("Top-level promoted logit decomposition failed")

contrib_rows = []
for name, arr, rawp, weight in [
    ("Stage15A physiology parent", L_PHYS, P_PHYS, W_PHYS),
    ("Guide-primary QML parent", L_QML, P_QML, W_QML),
]:
    contrib_rows.append({
        "parent": name,
        "fixed_weight": float(weight),
        "mean_abs_weighted_logit_contribution": float(np.mean(np.abs(arr))),
        "mean_signed_weighted_logit_contribution": float(np.mean(arr)),
        "mean_abs_contribution_predicted_apnea": float(np.mean(np.abs(arr[PRED_FINAL==1]))),
        "mean_abs_contribution_predicted_normal": float(np.mean(np.abs(arr[PRED_FINAL==0]))),
        "mean_parent_probability": float(np.mean(rawp)),
    })

CONTRIB_DF = pd.DataFrame(contrib_rows).sort_values(
    "mean_abs_weighted_logit_contribution", ascending=False
).reset_index(drop=True)
atomic_csv(XAI_OUT / "PROMOTED_FINAL_PARENT_LOGIT_CONTRIBUTION.csv", CONTRIB_DF)

parent_disagree = np.abs(logit(P_PHYS) - logit(P_QML))
rep_idx = {
    "highest_final_probability": int(np.argmax(P_FINAL)),
    "lowest_final_probability": int(np.argmin(P_FINAL)),
    "closest_to_threshold": int(np.argmin(np.abs(P_FINAL-THR))),
    "maximum_parent_logit_disagreement": int(np.argmax(parent_disagree)),
    "largest_abs_phys_weighted_logit": int(np.argmax(np.abs(L_PHYS))),
    "largest_abs_qml_weighted_logit": int(np.argmax(np.abs(L_QML))),
}
rep_rows = []
for tag, i in rep_idx.items():
    rep_rows.append({
        "case": tag,
        "row": int(i),
        "uid": UID[i],
        "record_name": REC[i],
        "epoch_idx": int(EP[i]),
        "physiology_probability": float(P_PHYS[i]),
        "qml_probability": float(P_QML[i]),
        "physiology_weighted_logit": float(L_PHYS[i]),
        "qml_weighted_logit": float(L_QML[i]),
        "fused_probability": float(P_FINAL[i]),
        "prediction": int(PRED_FINAL[i]),
        "parent_abs_logit_disagreement": float(parent_disagree[i]),
    })
REP_DF = pd.DataFrame(rep_rows)
atomic_csv(XAI_OUT / "PROMOTED_FINAL_REPRESENTATIVE_CASES.csv", REP_DF)

display(CONTRIB_DF)
display(REP_DF)


In [ ]:

# Cell 5 — Stage 09 refresh: Stage15A classical-parent model feature importance (NO training)

# Loading the already-frozen CatBoost model files requires catboost to be importable.
try:
    import joblib
    import catboost
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost==1.2.8", "joblib"])
    import joblib
    import catboost

schema = json.loads(A15_SCHEMA.read_text())
FEATURE_NAMES = list(schema["feature_names"])
if len(FEATURE_NAMES) != 379:
    raise RuntimeError(f"Expected 379 Stage15A feature names, got {len(FEATURE_NAMES)}")

phys_manifest = json.loads(PHYS_FREEZE.read_text())
members = list(phys_manifest["full_learning_selected_rule"]["members"])
if members != ["cat_group_d8"]:
    raise RuntimeError(f"Unexpected Stage15A final member rule: {members}")

model_files = sorted(PHYS_MODELS.glob("cat_group_d8__seed*.joblib"))
if len(model_files) != 5:
    raise RuntimeError(f"Expected 5 frozen cat_group_d8 models, found {len(model_files)}")

IMP = []
model_hash_rows = []
for p in model_files:
    model = joblib.load(p)
    imp = np.asarray(model.feature_importances_, np.float64)
    if imp.shape != (379,) or not np.isfinite(imp).all():
        raise RuntimeError(f"Invalid feature importance vector from {p.name}: {imp.shape}")
    IMP.append(imp)
    model_hash_rows.append({
        "model_file": p.name,
        "sha256": sha256_file(p),
        "feature_count": int(len(imp)),
    })

IMP = np.stack(IMP, axis=0)
mean_imp = IMP.mean(axis=0)
std_imp = IMP.std(axis=0)

CLASSICAL_XAI = pd.DataFrame({
    "feature_index": np.arange(379, dtype=int),
    "feature_name": FEATURE_NAMES,
    "mean_catboost_feature_importance": mean_imp,
    "std_catboost_feature_importance_across_5_seeds": std_imp,
}).sort_values("mean_catboost_feature_importance", ascending=False).reset_index(drop=True)

atomic_csv(XAI_OUT / "STAGE15A_CLASSICAL_PARENT_FEATURE_IMPORTANCE.csv", CLASSICAL_XAI)
atomic_csv(XAI_OUT / "STAGE15A_CLASSICAL_PARENT_MODEL_HASHES.csv", pd.DataFrame(model_hash_rows))

print("Frozen Stage15A models loaded:", len(model_files))
print("No model fit/training occurred.")
display(CLASSICAL_XAI.head(30))


In [ ]:

# Cell 6 — Stage 09 refresh: bind unchanged QML-component XAI by exact hashes, then close Stage 09

inherited_rows = []
for p in sorted(OLD_XAI.iterdir()):
    if p.is_file():
        inherited_rows.append({
            "artifact": p.name,
            "source_path": str(p),
            "sha256": sha256_file(p),
            "scope": "unchanged guide-primary QML parent component evidence",
        })

INHERITED_XAI = pd.DataFrame(inherited_rows)
atomic_csv(XAI_OUT / "INHERITED_QML_COMPONENT_XAI_ARTIFACT_INDEX.csv", INHERITED_XAI)

xai_manifest = {
    "schema": "QML_SleepNet_XAI_CAUSAL_INTERPRETABILITY_90P8627_PROMOTED_FINAL_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "roadmap_stage": "Stage 09 — Interpretability and Causal Analysis (promoted-final refresh)",
    "promoted_final_model": "25% Stage15A physiology + 75% guide-primary QML, fixed logit fusion",
    "promoted_final_prediction_sha256": EXPECTED_FUSED_SHA,
    "locked_official_x_accuracy": M_FINAL["accuracy"],
    "locked_official_x_accuracy_percent": 100*M_FINAL["accuracy"],
    "training_performed": False,
    "model_selection_performed": False,
    "predictions_modified": False,
    "threshold_tuning_performed": False,
    "fusion_weight_tuning_performed": False,
    "exact_top_level_explanation": {
        "method": "weighted logit decomposition",
        "weights": {"physiology": W_PHYS, "qml": W_QML},
        "artifact": "PROMOTED_FINAL_PARENT_LOGIT_CONTRIBUTION.csv",
        "reconstruction_max_abs_error": recon_max,
    },
    "classical_parent_explanation": {
        "method": "mean frozen CatBoost feature_importances_ across the five final all-learning seed models",
        "artifact": "STAGE15A_CLASSICAL_PARENT_FEATURE_IMPORTANCE.csv",
        "model_hashes": "STAGE15A_CLASSICAL_PARENT_MODEL_HASHES.csv",
        "limitation": "CatBoost model importance is not causal evidence and is not SHAP.",
    },
    "qml_parent_xai_reuse": {
        "source_qml_prediction_sha256": EXPECTED_QML_SHA,
        "source_xai_manifest": str(OLD_XAI_MANIFEST),
        "source_manifest_sha256": sha256_file(OLD_XAI_MANIFEST),
        "artifact_index": "INHERITED_QML_COMPONENT_XAI_ARTIFACT_INDEX.csv",
        "reason_valid": "The QML parent used by the promoted fusion is byte-for-byte the same frozen 90.1148% QML system.",
    },
    "representative_cases": "PROMOTED_FINAL_REPRESENTATIVE_CASES.csv",
    "important_limitations": [
        "Top-level contribution is exact logit decomposition, not a causal attribution.",
        "Stage15A feature importance is model importance, not clinical causality.",
        "Inherited QML causal/SHAP/latent/ECG explanations explain the unchanged QML parent, not the Stage15A parent.",
        "ACE remains model-based causal evidence under Stage05 assumptions, not clinical proof.",
        "No quantum-advantage claim.",
    ],
    "next_roadmap_stage": "Stage 10 — Model Robustness (promoted-final refresh)",
}
atomic_json(XAI_OUT / "XAI_PROMOTED_FINAL_MANIFEST.json", xai_manifest)

print("="*110)
print("STAGE 09 PROMOTED-FINAL XAI REFRESH — COMPLETE")
print("="*110)
print("Final SHA:", EXPECTED_FUSED_SHA)
print("Model/predictions changed: NO")
print("Evidence:", XAI_OUT)


In [ ]:

# Cell 7 — Stage 10 refresh: promoted top-level ablation + CV/unseen evidence

systems = {
    "PROMOTED_FINAL_PHYS25_QML75": P_FINAL,
    "PHYSIOLOGY_PARENT_ONLY": P_PHYS,
    "QML_PARENT_ONLY": P_QML,
    # Fixed-weight parent neutralizations: the removed parent contributes logit(0.5)=0.
    "NEUTRALIZE_PHYS_PARENT_KEEP_QML_WEIGHT75": sigmoid(W_QML*logit(P_QML)),
    "NEUTRALIZE_QML_PARENT_KEEP_PHYS_WEIGHT25": sigmoid(W_PHYS*logit(P_PHYS)),
}

abl_rows = []
for name, p in systems.items():
    m = metrics(Y, p, THR)
    abl_rows.append({
        "system": name,
        **m,
        "accuracy_delta_vs_promoted_final_pp": 100*(m["accuracy"]-M_FINAL["accuracy"]),
    })

ABL_DF = pd.DataFrame(abl_rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
atomic_csv(S10_OUT / "STAGE10_PROMOTED_PARENT_ABLATION.csv", ABL_DF)

dev_winner = freeze["winning_fusion"]
cv_unseen = {
    "promoted_final_prediction_sha256": EXPECTED_FUSED_SHA,
    "development_selection_unit": "strict development OOF; fixed candidate family only",
    "development_winner": dev_winner["candidate"],
    "development_oof_accuracy": float(dev_winner["accuracy"]),
    "development_oof_balanced_accuracy": float(dev_winner["balanced_accuracy"]),
    "development_oof_f1": float(dev_winner["f1"]),
    "development_oof_auprc": float(dev_winner["auprc"]),
    "strict_group_bootstrap_artifact": str(FUSION_BOOTSTRAP),
    "official_unseen_x_accuracy": float(M_FINAL["accuracy"]),
    "official_unseen_x_n": int(M_FINAL["n"]),
    "notes": [
        "No cross-validation is rerun in this refresh.",
        "The promoted fusion was selected on strict development OOF before its fused x score was opened.",
        "Official x evidence is historical/post-hoc at the project level because parent x results were already known before the fusion idea was conceived.",
    ],
}
atomic_json(S10_OUT / "STAGE10_PROMOTED_CV_UNSEEN_SUMMARY.json", cv_unseen)

display(ABL_DF)
print(json.dumps(cv_unseen, indent=2))


In [ ]:

# Cell 8 — Stage 10 refresh: domain shift/generalization and artifact-proxy metrics on promoted final

taskc = pd.read_csv(TASKC_RECORDS)
taskc["record_name"] = taskc["record_name"].astype(str)

minute = pd.DataFrame({
    "uid": UID,
    "record_name": REC,
    "epoch_idx": EP,
    "y": Y,
    "p": P_FINAL,
})
minute = minute.merge(
    taskc[[
        "record_name", "scorable_minutes", "official_annotated_minutes_full",
        "true_apnea_minutes_full", "taskc_true_class_full"
    ]],
    on="record_name", how="left", validate="many_to_one"
)
if minute["taskc_true_class_full"].isna().any():
    raise RuntimeError("Task-C metadata merge incomplete")

minute["apnea_burden"] = (
    minute["true_apnea_minutes_full"] /
    minute["official_annotated_minutes_full"].clip(lower=1)
)

record_meta = (
    minute[["record_name","apnea_burden","scorable_minutes"]]
    .drop_duplicates("record_name")
    .sort_values("record_name")
    .reset_index(drop=True)
)

def robust_record_qbin(series, prefix):
    codes = pd.qcut(series.astype(float), q=4, labels=False, duplicates="drop")
    n_bins = int(codes.max()) + 1 if codes.notna().any() else 0
    return codes.map(lambda x: f"{prefix}{int(x)+1}_of_{n_bins}" if pd.notna(x) else "NA")

record_meta["burden_quartile"] = robust_record_qbin(record_meta["apnea_burden"], "Q")
record_meta["length_quartile"] = robust_record_qbin(record_meta["scorable_minutes"], "Q")

minute = minute.merge(
    record_meta[["record_name","burden_quartile","length_quartile"]],
    on="record_name", how="left", validate="many_to_one"
)

gen_rows = []
for col in ["taskc_true_class_full","burden_quartile","length_quartile"]:
    for val, g in minute.groupby(col, observed=True):
        gen_rows.append({"grouping": col, "group": str(val), **metrics(g["y"], g["p"], THR)})
GEN_DF = pd.DataFrame(gen_rows)
atomic_csv(S10_OUT / "STAGE10_PROMOTED_GENERALIZATION_DOMAIN_SHIFT.csv", GEN_DF)

# Reuse exact deterministic Stage02 quality proxy rows/mask definition.
quality = pd.read_csv(OLD_S10_QUALITY)
quality["uid"] = quality["uid"].astype(str)
if len(quality) != len(UID) or not np.array_equal(quality["uid"].to_numpy(), UID):
    raise RuntimeError("Inherited Stage10 quality-proxy UID alignment failure")

old_art = json.loads(OLD_S10_ARTIFACT.read_text())
diff_cut = float(old_art["diff_rms_q75"])
flat_cut = float(old_art["flat_fraction_q75"])

artifact_mask = (
    (quality["clip_fraction_preclip"].to_numpy() > 0) |
    (quality["diff_rms"].to_numpy() >= diff_cut) |
    (quality["flat_fraction"].to_numpy() >= flat_cut)
)

artifact_report = {
    "definition": old_art["definition"],
    "mask_reused_from_qml_stage10": True,
    "source_quality_proxy_sha256": sha256_file(OLD_S10_QUALITY),
    "n": int(artifact_mask.sum()),
    "fraction": float(artifact_mask.mean()),
    "diff_rms_q75": diff_cut,
    "flat_fraction_q75": flat_cut,
    "promoted_final_metrics": metrics(Y[artifact_mask], P_FINAL[artifact_mask], THR),
    "important_limitation": "Deterministic ECG-quality stress proxy; not clinically annotated artifact truth.",
}
atomic_json(S10_OUT / "STAGE10_PROMOTED_ARTIFACT_SEGMENT_REPORT.json", artifact_report)

display(GEN_DF)
print(json.dumps(artifact_report, indent=2))


In [ ]:

# Cell 9 — Stage 10 refresh: propagate existing Stage06 perturbation caches through the promoted fusion

# Load the unchanged canonical-QML frozen artifact for branch/HMM parameters.
qfz = np.load(QML_X, allow_pickle=False)
Q_UID = np.asarray(qfz["test_uids"]).astype(str)
if not np.array_equal(Q_UID, UID):
    raise RuntimeError("QML parent UID order differs from promoted final")

BRIDGE_HMM = np.asarray(qfz["bridge_hmm_posterior"], np.float64)
QT_HMM = np.asarray(qfz["qt_hmm_posterior"], np.float64)
S6_HMM_CLEAN = np.asarray(qfz["stage06_hmm_posterior"], np.float64)

T_S6 = float(np.asarray(qfz["temperature_stage06"]).item())
CLASS_PRIOR = np.asarray(qfz["class_prior"], np.float64)
PI = np.asarray(qfz["initial_state_prior"], np.float64)
A = np.asarray(qfz["transition_matrix"], np.float64)
HMM_LAMBDA = float(np.asarray(qfz["hmm_lambda"]).item())

def temperature_scale(prob, T):
    return np.clip(sigmoid(logit(prob)/float(T)), EPS, 1-EPS)

def contiguous_segments(indices, uids):
    indices = np.asarray(indices, np.int64)
    rec = np.asarray([u.rsplit(":",1)[0] for u in uids])
    ep = np.asarray([int(u.rsplit(":",1)[1]) for u in uids], np.int64)
    out = []
    for r in np.unique(rec[indices]):
        rr = indices[rec[indices] == r]
        order = rr[np.argsort(ep[rr])]
        ee = ep[order]
        cuts = [0] + (np.where(np.diff(ee) != 1)[0] + 1).tolist() + [len(order)]
        out.extend(order[a:b] for a,b in zip(cuts[:-1], cuts[1:]) if b>a)
    return out

def _logsumexp(v):
    v = np.asarray(v, np.float64)
    m = np.max(v)
    return float(m + np.log(np.exp(v-m).sum()))

def forward_backward(prob, prior, pi, A):
    p = np.clip(np.asarray(prob, float), EPS, 1-EPS)
    post = np.column_stack([1-p, p])
    logE = np.log(post) - HMM_LAMBDA*np.log(np.clip(prior, EPS, 1))[None,:]
    logE -= np.max(logE, axis=1, keepdims=True)
    n = len(p)
    lpi = np.log(np.clip(pi, EPS, 1))
    lA = np.log(np.clip(A, EPS, 1))
    alpha = np.full((n,2), -np.inf)
    beta = np.full((n,2), -np.inf)
    alpha[0] = lpi + logE[0]
    for t in range(1,n):
        for s in range(2):
            alpha[t,s] = logE[t,s] + _logsumexp(alpha[t-1] + lA[:,s])
    ll = _logsumexp(alpha[-1])
    beta[-1] = 0.0
    for t in range(n-2,-1,-1):
        for s in range(2):
            beta[t,s] = _logsumexp(lA[s] + logE[t+1] + beta[t+1])
    gamma = np.exp(alpha + beta - ll)
    gamma /= gamma.sum(axis=1, keepdims=True)
    return gamma[:,1]

def decode_external(raw_prob, uids, T, prior, pi, A):
    pcal = temperature_scale(raw_prob, T)
    idx = np.arange(len(uids), dtype=np.int64)
    out = np.full(len(uids), np.nan, np.float64)
    for seg in contiguous_segments(idx, uids):
        out[seg] = forward_backward(pcal[seg], prior, pi, A)
    if not np.isfinite(out).all():
        raise RuntimeError("HMM decode incomplete")
    return out

def load_cached_raw(condition):
    full = np.full(len(UID), np.nan, np.float64)
    cps = sorted(OLD_S10_CACHE.glob(f"{condition}__x*.npz"))
    if len(cps) != 35:
        raise RuntimeError(f"{condition}: expected 35 inherited cache files, found {len(cps)}")
    touched = np.zeros(len(UID), dtype=bool)
    for cp in cps:
        z = np.load(cp, allow_pickle=False)
        rows = np.asarray(z["rows"], np.int64)
        raw = np.asarray(z["raw_probability"], np.float64)
        if len(rows) != len(raw) or np.any(rows < 0) or np.any(rows >= len(UID)):
            raise RuntimeError(f"{condition}: invalid cache {cp.name}")
        if touched[rows].any():
            raise RuntimeError(f"{condition}: overlapping cache rows")
        full[rows] = raw
        touched[rows] = True
    if not touched.all() or not np.isfinite(full).all():
        raise RuntimeError(f"{condition}: incomplete inherited raw cache")
    return full

old_pert = pd.read_csv(OLD_S10_PERT).set_index("condition")
conditions = ["SNR20_DB", "SNR10_DB", "CENTRAL_45S", "CENTRAL_30S"]
rows = []

for condition in conditions:
    raw = load_cached_raw(condition)
    s6_hmm = decode_external(raw, UID, T_S6, CLASS_PRIOR, PI, A)
    qml_pert = equal_logit_mean_n(BRIDGE_HMM, QT_HMM, s6_hmm)

    # Verify we exactly reconstruct the previously completed QML Stage10 condition.
    qml_m = metrics(Y, qml_pert, THR)
    old_acc = float(old_pert.loc[condition, "final_ensemble_accuracy"])
    if abs(qml_m["accuracy"] - old_acc) > 1e-12:
        raise RuntimeError(
            f"{condition}: inherited QML Stage10 reconstruction mismatch "
            f"{qml_m['accuracy']} vs {old_acc}"
        )

    # Promoted final: physiology parent remains frozen clean; perturbed QML parent is propagated
    # through the exact fixed 25/75 logit fusion.
    promoted_pert = fixed_logit_fusion(P_PHYS, qml_pert, W_PHYS)
    pm = metrics(Y, promoted_pert, THR)

    rows.append({
        "condition": condition,
        "scope": "Stage06 temporal-channel perturbation propagated through unchanged QML parent; physiology parent held clean",
        "qml_parent_perturbed_accuracy": qml_m["accuracy"],
        "qml_parent_perturbed_balanced_accuracy": qml_m["balanced_accuracy"],
        "qml_parent_perturbed_f1": qml_m["f1"],
        "qml_parent_perturbed_auroc": qml_m["auroc"],
        "promoted_final_accuracy": pm["accuracy"],
        "promoted_final_balanced_accuracy": pm["balanced_accuracy"],
        "promoted_final_f1": pm["f1"],
        "promoted_final_auroc": pm["auroc"],
        "promoted_final_auprc": pm["auprc"],
        "promoted_final_accuracy_delta_vs_clean_pp": 100*(pm["accuracy"]-M_FINAL["accuracy"]),
    })

PERT_DF = pd.DataFrame(rows)
atomic_csv(S10_OUT / "STAGE10_PROMOTED_PERTURBATION_ROBUSTNESS.csv", PERT_DF)

display(PERT_DF)
print(
    "\nIMPORTANT SCOPE: These are inherited Stage06-channel perturbations propagated through the "
    "promoted 25/75 fusion. They are NOT claimed as full dual-parent raw-ECG perturbation tests."
)


In [ ]:

# Cell 10 — Stage 10 refresh: bind inherited QML8/causal16 sensitivity; unsupported items; final manifest

# The QML parent itself is unchanged, so component-level QML8/causal16 sensitivity remains valid.
sens = pd.read_csv(OLD_S10_SENS)
atomic_csv(S10_OUT / "INHERITED_QML_COMPONENT_FEATURE_ZEROOUT_SENSITIVITY.csv", sens)

unsupported_old = json.loads(OLD_S10_UNSUPPORTED.read_text())
unsupported = {
    **unsupported_old,
    "full_dual_parent_raw_ecg_perturbation": {
        "status": "NOT_CLAIMED_IN_THIS_REFRESH",
        "reason": (
            "The completed Stage10 perturbation cache changes only the Stage06 temporal branch at the "
            "post-Stage02 ECG input boundary. The promoted Stage15A physiology parent would require a "
            "separate deterministic re-extraction of its full 379-feature bank under each perturbed signal. "
            "This refresh does not fabricate or approximate that missing end-to-end pathway."
        ),
        "action": (
            "Report the promoted perturbation results as Stage06/QML-channel stress propagated through the "
            "promoted fusion; do not describe them as dual-parent end-to-end noise robustness."
        ),
    },
}
atomic_json(S10_OUT / "STAGE10_PROMOTED_UNSUPPORTED_OR_SCOPE_LIMITED_ITEMS.json", unsupported)

s10_manifest = {
    "schema": "QML_SleepNet_STAGE10_MODEL_ROBUSTNESS_90P8627_PROMOTED_FINAL_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "stage": "Stage 10 — Model Robustness (promoted-final refresh)",
    "primary_frozen_model": "25% Stage15A physiology + 75% guide-primary QML; fixed logit fusion",
    "primary_frozen_prediction_sha256": EXPECTED_FUSED_SHA,
    "locked_official_x_accuracy": M_FINAL["accuracy"],
    "training_performed": False,
    "model_selection_performed": False,
    "threshold_tuning_performed": False,
    "ensemble_weight_tuning_performed": False,
    "headline_model_modified": False,
    "official_x_labels_role": "evaluation only; historical x exposure caveat retained",
    "stage10_1_cross_validation_unseen": {
        "status": "COMPLETE_BY_REUSE_AND_PROMOTED_REBIND",
        "artifact": "STAGE10_PROMOTED_CV_UNSEEN_SUMMARY.json",
    },
    "stage10_2_noise_and_temporal_coverage": {
        "status": "COMPLETE_FOR_PREVIOUSLY_DEFINED_STAGE06_QML_PERTURBATION_PATHWAY",
        "conditions": ["SNR20_DB","SNR10_DB","CENTRAL_45S","CENTRAL_30S"],
        "artifact": "STAGE10_PROMOTED_PERTURBATION_ROBUSTNESS.csv",
        "scope_limitation": (
            "Stage06 temporal branch perturbed; Bridge/QT and promoted Stage15A physiology parent held clean. "
            "No full dual-parent raw-ECG perturbation claim."
        ),
    },
    "stage10_2_artifact_segments": {
        "status": "COMPLETE_WITH_EXISTING_DETERMINISTIC_HEURISTIC_PROXY",
        "artifact": "STAGE10_PROMOTED_ARTIFACT_SEGMENT_REPORT.json",
    },
    "stage10_3_generalization": {
        "domain_shift": "COMPLETE on promoted clean predictions: Task-C class, apnea-burden quartile, record-length quartile",
        "variable_temporal_coverage": "COMPLETE for inherited Stage06-channel 45s/30s masking propagated through promoted fusion",
        "different_sleep_stages": "NOT EVALUABLE: no sleep-stage target in frozen project contract",
        "artifact": "STAGE10_PROMOTED_GENERALIZATION_DOMAIN_SHIFT.csv",
    },
    "stage10_4_ablation": {
        "promoted_parent_ablation": "COMPLETE",
        "artifact": "STAGE10_PROMOTED_PARENT_ABLATION.csv",
        "qml8_causal16_component_sensitivity": (
            "INHERITED from exact unchanged QML parent; component-level only, no claim that it is "
            "top-level promoted-final feature attribution"
        ),
        "component_sensitivity_artifact": "INHERITED_QML_COMPONENT_FEATURE_ZEROOUT_SENSITIVITY.csv",
    },
    "unsupported_or_scope_limited": "STAGE10_PROMOTED_UNSUPPORTED_OR_SCOPE_LIMITED_ITEMS.json",
    "outputs": [
        "STAGE10_PROMOTED_CV_UNSEEN_SUMMARY.json",
        "STAGE10_PROMOTED_PARENT_ABLATION.csv",
        "STAGE10_PROMOTED_GENERALIZATION_DOMAIN_SHIFT.csv",
        "STAGE10_PROMOTED_ARTIFACT_SEGMENT_REPORT.json",
        "STAGE10_PROMOTED_PERTURBATION_ROBUSTNESS.csv",
        "INHERITED_QML_COMPONENT_FEATURE_ZEROOUT_SENSITIVITY.csv",
        "STAGE10_PROMOTED_UNSUPPORTED_OR_SCOPE_LIMITED_ITEMS.json",
    ],
    "next": "Stage 11 — Final Evaluation Protocol / statistical comparison",
}
atomic_json(S10_OUT / "STAGE10_PROMOTED_FINAL_MANIFEST.json", s10_manifest)

print("="*110)
print("QML-SLEEPNET PROMOTED FINAL STAGE 09 + STAGE 10 REFRESH — COMPLETE")
print("="*110)
print("Promoted final SHA:", EXPECTED_FUSED_SHA)
print("Official-x accuracy:", 100*M_FINAL["accuracy"])
print("Training performed: NO")
print("Model selection performed: NO")
print("Threshold/weight/HMM/calibration tuning: NO")
print("Stage09 evidence:", XAI_OUT)
print("Stage10 evidence:", S10_OUT)
print("NEXT: Stage 11 — Final Evaluation Protocol / statistical comparison")



## Acceptance rule

Only close the refresh if the final cell prints:

- `PROMOTED FINAL STAGE 09 + STAGE 10 REFRESH — COMPLETE`
- promoted final SHA `063a017e...`
- official-x accuracy `90.86270871985158`
- all training/tuning/model-selection flags `NO`
- `NEXT: Stage 11 — Final Evaluation Protocol / statistical comparison`

If any fail-closed check raises an error, **do not patch around it**. Send the traceback and the relevant manifest/output back for audit.

This notebook is an **evaluation/explanation refresh only**. It must never trigger another metric search.
